In [73]:
# packages
import pandas as pd
from mod02_build_bot_predictor import train_model

### Define a function to extract predictions from the model

In [74]:
def predict_bot(df, model=None):
    """
    Predict whether each account is a bot (1) or human (0).
    """
    if model is None:
        model = train_model()

    preds = model.predict(df)
    return pd.Series(preds, index=df.index)

### Define a function to evaluate model error

In [75]:
def confusion_matrix_and_metrics(y_true, y_pred):
    """
    Computes confusion matrix and common error rates for binary classification.

    Assumes labels:
      0 = negative class
      1 = positive class

    Returns:
      dict with:
        tn, fp, fn, tp
        misclassification_rate
        false_positive_rate
        false_negative_rate
    """
    tn = fp = fn = tp = 0

    for yt, yp in zip(y_true, y_pred):
        if yt == 0 and yp == 0:
            tn += 1
        elif yt == 0 and yp == 1:
            fp += 1
        elif yt == 1 and yp == 0:
            fn += 1
        elif yt == 1 and yp == 1:
            tp += 1
        else:
            raise ValueError("Labels must be 0 or 1")

    total = tn + fp + fn + tp

    misclassification_rate = (fp + fn) / total if total > 0 else 0.0
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "misclassification_rate": misclassification_rate,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
    }


### Load the data

In [76]:
TRAIN_PATH = "mod02_data/train.csv"
train = pd.read_csv(TRAIN_PATH)

TEST_PATH = "mod02_data/test.csv"
test = pd.read_csv(TEST_PATH)

### Format the data by independent vs. dependent variables

In [77]:
X_train = train.drop(columns=["is_bot"])
y_train = train['is_bot']

X_test = test.drop(columns=["is_bot"])
y_test = test['is_bot']

### Build the model on training data

In [78]:
model = train_model(X_train, y_train)

### Get the model predictions on training and test data

In [79]:
y_pred_train = predict_bot(X_train, model)
y_pred_test = predict_bot(X_test, model)

### Check results on the training set (data used to build the model)

In [80]:
confusion_matrix_and_metrics(y_train, y_pred_train)

{'tp': 112,
 'tn': 2595,
 'fp': 42,
 'fn': 251,
 'misclassification_rate': 0.09766666666666667,
 'false_positive_rate': 0.015927189988623434,
 'false_negative_rate': 0.6914600550964187}

### Check results on the test set (new data not yet seen by the model)

In [81]:
confusion_matrix_and_metrics(y_test, y_pred_test)

{'tp': 33,
 'tn': 859,
 'fp': 15,
 'fn': 93,
 'misclassification_rate': 0.108,
 'false_positive_rate': 0.017162471395881007,
 'false_negative_rate': 0.7380952380952381}

# Discussion Questions

### Based on the misclassification rate of your model, discuss your confidence in the ability to predict a bot. 

I am not very confident in this model's ability to predict a bot. By lowering the max_depth from 8 to 3, the misclassification rate is at 10.8% which is considerably really low. However, the false negative rate is at 73.8% meaning out of 126 bots, the model could only predict 33 of them being bots. 

### What are potential ramifications of false positives from the model?

The false positive rate is 1.7% meaning it falsely identified 37 users as bots. This could cause problems for users who needs access or help. Not only that, this could get bad reviews for the company which can lead to potential customer loss.

### What are potential ramifications of false negatives from the model?

The false negative is 73.8% meaning it falsely identified 93 bots as users which is very dangerous. This is a huge cybersecurity issue for the company. If a company were to let bots into their database, there's a chance that those bots can send many spam emails to the company and if the company were to respond to those bots, the people behind the bots could easily steal information from the company. I knew a coworker who responded to a phishing email which lead to loss of database from the company. 